# Exam 2025

In [12]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")


E=12 # No of exams
R=3 # No of rooms
T=4 # No of timeslots

# No of students in each exam
ExamStudents=[ 40 97 100 18 73 55 96 82 82 55 38 93] 

# No of students who can take an exam in this room in
# one timeslot (number of seats of seats)
RoomCap=[ 96 88 190] 

# Cost of using a room in a timeslot:
# RoomCost[r,t], i.e. RoomCost[2,3]=3491
RoomCost=[
 3510  3563  3511  3582 ;
 3473  3420  3491  3550 ;
 5407  5464  5522  5463 ]

# Extra penalty having exam e in room r:
# ExamTimeslotPenalty[E,T], i.e. ExamTimeslotPenalty[3,2]=89
ExamTimeslotPenalty=[
 87  20  30  84 ;
 43  43  47  50 ;
 96  89  48  48 ;
 83  48  90  82 ;
 66  76  71  38 ;
 75  57  62  56 ;
 26  24  52  67 ;
 90  58  15  63 ;
 52  85  80  12 ;
 11  82  57  89 ;
 40  38  58  56 ;
 18  13  37  75 ]

# ExamRoomPenalty[E,R], penalty for having exam e in room r:
# ExamRoomPenalty[2,3]=37
ExamRoomPenalty=[
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ;
 96  97  37 ]

# Incidence[E,E] matrix:
# Collision[e1,e2]=0 means that exam e1 and exam e2
# cannot take place in the same timeslot
Collision=[
 1  0  1  0  1  0  0  0  0  1  1  0 ;
 0  1  1  0  1  1  1  1  1  0  0  0 ;
 1  1  1  1  1  0  0  0  1  0  1  0 ;
 0  0  1  1  0  1  0  1  1  1  1  1 ;
 1  1  1  0  1  1  0  1  0  0  1  1 ;
 0  1  0  1  1  1  1  0  1  1  0  1 ;
 0  1  0  0  0  1  1  0  0  1  0  1 ;
 0  1  0  1  1  0  0  1  0  1  0  1 ;
 0  1  1  1  0  1  0  0  1  0  1  1 ;
 1  0  0  1  0  1  1  1  0  1  0  0 ;
 1  0  1  1  1  0  0  0  1  0  1  1 ;
 0  0  0  1  1  1  1  1  1  0  1  1 ]





   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`


12×12 Matrix{Int64}:
 1  0  1  0  1  0  0  0  0  1  1  0
 0  1  1  0  1  1  1  1  1  0  0  0
 1  1  1  1  1  0  0  0  1  0  1  0
 0  0  1  1  0  1  0  1  1  1  1  1
 1  1  1  0  1  1  0  1  0  0  1  1
 0  1  0  1  1  1  1  0  1  1  0  1
 0  1  0  0  0  1  1  0  0  1  0  1
 0  1  0  1  1  0  0  1  0  1  0  1
 0  1  1  1  0  1  0  0  1  0  1  1
 1  0  0  1  0  1  1  1  0  1  0  0
 1  0  1  1  1  0  0  0  1  0  1  1
 0  0  0  1  1  1  1  1  1  0  1  1

In [13]:
using JuMP, HiGHS
########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

max_studenst = 100

########## ---------- variables ---------- ##########
# x is the no. of student assigned to each exam, room, timeslot
@variable(model, x[1:E, 1:R, 1:T] >= 0) 
# Indicate wheter students has been assigned
@variable(model, y[1:E, 1:R, 1:T], Bin) 
# Room usage at time t
@variable(model, RoomTime[1:R, 1:T], Bin)
# Exam in timeslot
@variable(model, ExamTimeslot[1:E, 1:T], Bin)
# Exam in Room
@variable(model, ExamRoom[1:E, 1:R], Bin)




########## ---------- Objectives ---------- ##########
@objective(model, Min, 
    sum(RoomCost[r,t] * RoomTime[r,t] for r in 1:R, t in 1:T) +
    sum(ExamTimeslotPenalty[e,t] * ExamTimeslot[e,t] for e in 1:E, t in 1:T) + 
    sum(ExamRoomPenalty[e,r] * ExamRoom[e,r] for e in 1:E, r in 1:R)
)

########## ---------- Constraint ---------- ##########
# First thing we need to do is set the binary variables
# We use a large big-M
@constraint(model, [e in 1:E, r in 1:R, t in 1:T],
    x[e,r,t] <= max_studenst * y[e,r,t]
)

# Then these needs to be dependt on y st. the model tries to minimize y
@constraint(model, [r in 1:R, t in 1:T],
    sum(y[e,r,t] for e in 1:E) <= E * RoomTime[r,t]
)
@constraint(model, [e in 1:E, t in 1:T],
    sum(y[e,r,t] for r in 1:R) <= R * ExamTimeslot[e,t]
)
@constraint(model, [e in 1:E, r in 1:R],
    sum(y[e,r,t] for t in 1:T) <= T * ExamRoom[e,r]
)

# We make sure that each student attend their exam
@constraint(model, [e in 1:E],
   sum(x[e,r,t] for r in 1:R, t in 1:T) == ExamStudents[e] 
)

# There cannot be more students attending an exam than the capcaity
@constraint(model, [r in 1:R, t in 1:T],
    sum(x[e,r,t]  for e in 1:E) <= RoomCap[r] 
)

########## ---------- Optimize ---------- ##########
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println(value.(RoomTime))
println(value.(ExamTimeslot))
println(value.(ExamRoom))

println("\n Student dist per timeslot")
for e in 1:E
    println("\nExam", e)
    for r in 1:R
        println("x :", value.(x[e,r,:]))
        println("y :", value.(y[e,r,:]))
    end
end





Optimal solution:
z = 26233.999999999993
[0.0 -0.0 -4.764707147349629e-16 0.0; 0.0 1.0 0.0 0.0; 1.0 1.0 1.0 1.0]
[-0.0 1.0000000000000002 0.0 0.0; -1.0343158174157335e-13 -0.0 1.0 -0.0; -0.0 0.0 -4.3129944060638083e-13 1.0000000000000113; 0.0 1.0000000000000002 0.0 0.0; 1.0 -0.0 -0.0 -0.0; 3.425192422684648e-12 1.0 -0.0 -3.425192422684648e-12; -5.34387349186242e-13 1.0000000000003129 -0.0 -0.0; 0.0 -0.0 1.0 -0.0; -1.314852195764583e-12 0.0 0.0 1.0000000000023388; 1.0 -2.501914225155361e-13 7.505742675466083e-13 0.0; 0.0 1.0 -0.0 -0.0; 1.000000000000326 1.0 -0.0 -0.0]
[0.0 0.0 1.0; 0.0 0.0 1.0; -0.0 -0.0 1.0; 0.0 -0.0 1.0; 0.0 0.0 1.0; -0.0 -0.0 1.0; 0.0 1.0 1.0; -0.0 -0.0 0.9999999999999999; 0.0 -0.0 1.0; -0.0 0.0 0.9999999999992493; -0.0 -0.0 1.0; 0.0 0.0 1.0]

 Student dist per timeslot

Exam1
x :[-0.0, -0.0, 0.0, -0.0]
y :[0.0, 0.0, 0.0, 0.0]
x :[-0.0, -0.0, -0.0, 0.0]
y :[0.0, -0.0, 0.0, 0.0]
x :[-0.0, 40.0, -0.0, -0.0]
y :[-0.0, 1.0, -0.0, 0.0]

Exam2
x :[0.0, 0.0, 0.0, 0.0]
y :[0

# part 2

In [21]:
using JuMP, HiGHS
########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

max_studenst = 100

########## ---------- variables ---------- ##########
# x is the no. of student assigned to each exam, room, timeslot
@variable(model, x[1:E, 1:R, 1:T] >= 0) 
# Indicate wheter students has been assigned
@variable(model, y[1:E, 1:R, 1:T], Bin) 
# Room usage at time t
@variable(model, RoomTime[1:R, 1:T], Bin)
# Exam in timeslot
@variable(model, ExamTimeslot[1:E, 1:T], Bin)
# Exam in Room
@variable(model, ExamRoom[1:E, 1:R], Bin)




########## ---------- Objectives ---------- ##########
@objective(model, Min, 
    sum(RoomCost[r,t] * RoomTime[r,t] for r in 1:R, t in 1:T) +
    sum(ExamTimeslotPenalty[e,t] * ExamTimeslot[e,t] for e in 1:E, t in 1:T) + 
    sum(ExamRoomPenalty[e,r] * ExamRoom[e,r] for e in 1:E, r in 1:R)
)

########## ---------- Constraint ---------- ##########
# First thing we need to do is set the binary variables
# We use a large big-M
@constraint(model, [e in 1:E, r in 1:R, t in 1:T],
    x[e,r,t] <= max_studenst * y[e,r,t]
)

# Then these needs to be dependt on y st. the model tries to minimize y
@constraint(model, [r in 1:R, t in 1:T],
    sum(y[e,r,t] for e in 1:E) <= E * RoomTime[r,t]
)
@constraint(model, [e in 1:E, t in 1:T],
    sum(y[e,r,t] for r in 1:R) <= R * ExamTimeslot[e,t]
)
@constraint(model, [e in 1:E, r in 1:R],
    sum(y[e,r,t] for t in 1:T) <= T * ExamRoom[e,r]
)

# We make sure that each student attend their exam
@constraint(model, [e in 1:E],
   sum(x[e,r,t] for r in 1:R, t in 1:T) == ExamStudents[e] 
)

# There cannot be more students attending an exam than the capcaity
@constraint(model, [r in 1:R, t in 1:T],
    sum(x[e,r,t]  for e in 1:E) <= RoomCap[r] 
)

# # Some exams cannot take place at the same time
@constraint(model, [e1 in 1:E, e2 in 1:E, t in 1:T; Collision[e1, e2] == 1],
    ExamTimeslot[e1, t]  <= ExamTimeslot[e2, t]
)


########## ---------- Optimize ---------- ##########
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println(value.(RoomTime))
println(value.(ExamTimeslot))
println(value.(ExamRoom))

println("\n Student dist per timeslot")
for e in 1:E
    println("\nExam", e)
    for r in 1:R
        println("x :", value.(x[e,r,:]))
        println("y :", value.(y[e,r,:]))
    end
end





Optimal solution:
z = 28467.0
[0.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 1.0 1.0 1.0 1.0]
[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0]
[-0.0 -0.0 1.000000000000001; -0.0 -0.0 1.0000000000000002; -0.0 -0.0 1.0000000000000007; -0.0 -0.0 1.0; -0.0 1.0 -0.0; 0.0 -0.0 1.0; -0.0 0.0 1.0; -0.0 -0.0 1.0; -0.0 -0.0 1.0; 0.0 -0.0 1.0; -0.0 -0.0 0.9999999999999998; -0.0 -0.0 1.0]

 Student dist per timeslot

Exam1
x :[-0.0, -0.0, -0.0, -0.0]
y :[-0.0, 0.0, -0.0, 0.0]
x :[-0.0, -0.0, 0.0, -0.0]
y :[0.0, 0.0, -0.0, 0.0]
x :[0.0, 36.99999999999993, 3.000000000000071, 0.0]
y :[-0.0, 1.0, 1.0, -0.0]

Exam2
x :[-0.0, -0.0, -0.0, 0.0]
y :[0.0, 0.0, -0.0, 0.0]
x :[0.0, -0.0, 0.0, -0.0]
y :[-0.0, 0.0, 0.0, 0.0]
x :[38.999999999999915, 0.0, 50.00000000000004, 8.000000000000028]
y :[1.0, -0.0, 1.0, 1.0]

Exam3
x :[-0.0, -0.0, 0.0, -0.0]
y :[0.0, 0.0, -0.0, 0